# Predicting 30-day hospital readmission

**Question:** Can the supplied admission/stay variables predict `readmit_30d`?

This notebook builds a leakage-aware binary-classification experiment. Because the same person has multiple stays, validation is grouped by `person_key`: a person can appear in training or validation/test, never both. This estimates performance for a **previously unseen person**, which is more conservative than randomly splitting stays.

**Prediction point:** discharge. `length_of_stay_days` and `n_medications` are used, so this model cannot be applied at admission. `person_key` is an identifier and is excluded. The test set is held out until model selection is complete.

> This is an exploratory modeling experiment, not a clinically validated decision tool. External/temporal validation, subgroup review, workflow design, and prospective monitoring would be required before use.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve
)
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', category=FutureWarning)

## 1. Load and understand the data

In [ ]:
DATA_PATH = Path('data.csv')
DICTIONARY_PATH = Path('columns.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run this notebook from the directory containing data.csv and columns.csv')

df = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
display(dictionary)
print(f'{len(df):,} stays, {df.person_key.nunique():,} people, {df.shape[1]} columns')
display(df.head())

In [ ]:
TARGET = 'readmit_30d'
ID = 'person_key'
required = set(dictionary['column'])
assert required == set(df.columns), 'Dictionary and data columns do not match'
assert df[TARGET].dropna().isin([0, 1]).all(), 'Target must be binary'

quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': df.isna().mean().mul(100).round(2),
    'unique_n': df.nunique(dropna=False),
})
display(quality)
print('Exact duplicate rows:', df.duplicated().sum())
print('Stays per person:')
display(df.groupby(ID).size().describe().to_frame('stays'))

# Basic plausibility flags. These are surfaced, not silently removed.
plausibility = pd.Series({
    'birth_year outside 1900..current year': (~df.birth_year.between(1900, pd.Timestamp.now().year)).sum(),
    'BMI outside 10..80': (~df.bmi.between(10, 80)).sum(),
    'negative length of stay': (df.length_of_stay_days < 0).sum(),
    'negative medication count': (df.n_medications < 0).sum(),
})
display(plausibility.to_frame('flagged_rows'))

In [ ]:
prevalence = df[TARGET].mean()
print(f'Readmission prevalence: {prevalence:.1%} ({df[TARGET].sum():,}/{len(df):,})')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=['#4C78A8', '#E45756'])
axes[0].set(title='Target counts', xlabel='Readmitted within 30 days', ylabel='Stays')

sex_rate = df.groupby('sex')[TARGET].agg(['mean', 'size']).sort_values('mean')
axes[1].bar(sex_rate.index.astype(str), sex_rate['mean'], color='#72B7B2')
axes[1].axhline(prevalence, color='black', ls='--', lw=1, label='overall')
axes[1].set(title='Observed rate by sex', xlabel='Sex', ylabel='Readmission rate')
axes[1].legend()

visit_rate = df.groupby('visit_number')[TARGET].agg(['mean', 'size'])
axes[2].plot(visit_rate.index, visit_rate['mean'], marker='o', color='#F58518')
axes[2].axhline(prevalence, color='black', ls='--', lw=1)
axes[2].set(title='Observed rate by visit number', xlabel='Visit number', ylabel='Readmission rate')
plt.tight_layout();

## 2. Leakage-safe split and preprocessing

The first fold of a shuffled 5-fold stratified group split is reserved as the final test set (about 20% of people). The remaining people are used for model selection with a separate 5-fold grouped cross-validation. Stratification is approximate because whole people, rather than individual rows, are allocated together.

`postcode_area` is treated as categorical: its numeric codes do not imply a continuous distance. Birth year is retained rather than converted to age because no admission/calendar year is supplied.

In [ ]:
y = df[TARGET].astype(int)
groups = df[ID]
X = df.drop(columns=[TARGET, ID])

outer_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_split.split(X, y, groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
g_train, g_test = groups.iloc[train_idx].copy(), groups.iloc[test_idx].copy()

assert set(g_train).isdisjoint(set(g_test))
split_summary = pd.DataFrame({
    'stays': [len(X_train), len(X_test)],
    'people': [g_train.nunique(), g_test.nunique()],
    'positive_rate': [y_train.mean(), y_test.mean()]
}, index=['train/model-selection', 'held-out test'])
display(split_summary.style.format({'positive_rate': '{:.1%}'}))

categorical_features = ['sex', 'postcode_area']
numeric_features = [c for c in X.columns if c not in categorical_features]
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features),
], sparse_threshold=0)

## 3. Model comparison

Average precision (AP) is the primary selection metric because only about one fifth of stays are positive and AP directly evaluates ranking quality for the positive class. ROC AUC, balanced accuracy, and Brier loss are included as complementary discrimination/calibration measures. The dummy model anchors what no-skill performance looks like.

In [ ]:
models = {
    'Dummy (prevalence)': DummyClassifier(strategy='prior'),
    'Logistic regression': LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE),
    'Random forest': RandomForestClassifier(
        n_estimators=400, min_samples_leaf=10, class_weight='balanced_subsample',
        n_jobs=1, random_state=RANDOM_STATE
    ),
    'Histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.06, l2_regularization=1.0, random_state=RANDOM_STATE
    ),
}
pipelines = {name: Pipeline([('prep', preprocessor), ('model', model)]) for name, model in models.items()}
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)
scoring = {
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'balanced_accuracy': 'balanced_accuracy',
    'neg_brier': 'neg_brier_score',
}
rows = []
for name, estimator in pipelines.items():
    scores = cross_validate(
        estimator, X_train, y_train, groups=g_train, cv=inner_cv,
        scoring=scoring, n_jobs=1, return_train_score=False
    )
    row = {'model': name}
    for metric in scoring:
        values = scores[f'test_{metric}']
        if metric == 'neg_brier':
            values = -values
            label = 'brier_loss'
        else:
            label = metric
        row[f'{label}_mean'] = values.mean()
        row[f'{label}_sd'] = values.std(ddof=1)
    rows.append(row)
cv_results = pd.DataFrame(rows).set_index('model').sort_values('average_precision_mean', ascending=False)
display(cv_results.style.format('{:.3f}'))

candidate_names = [name for name in models if not name.startswith('Dummy')]
best_name = cv_results.loc[candidate_names, 'average_precision_mean'].idxmax()
best_estimator = clone(pipelines[best_name])
print('Selected by mean grouped-CV average precision:', best_name)

## 4. Choose an operating threshold without using the test set

A probability threshold is a workflow choice, not an intrinsic model property. As an illustrative screening policy, the threshold below targets at least 70% recall in out-of-fold predictions from the training people, choosing the highest qualifying threshold. In practice, the target should be chosen from intervention capacity and the relative harm of missed versus unnecessary follow-up.

In [ ]:
oof_probability = cross_val_predict(
    best_estimator, X_train, y_train, groups=g_train, cv=inner_cv,
    method='predict_proba', n_jobs=1
)[:, 1]
oof_precision, oof_recall, thresholds = precision_recall_curve(y_train, oof_probability)
target_recall = 0.70
qualifying = np.flatnonzero(oof_recall[:-1] >= target_recall)
decision_threshold = thresholds[qualifying[-1]] if len(qualifying) else 0.5
oof_pred = (oof_probability >= decision_threshold).astype(int)
print(f'Training OOF threshold: {decision_threshold:.3f}')
print(f'OOF precision: {precision_score(y_train, oof_pred):.3f}')
print(f'OOF recall:    {recall_score(y_train, oof_pred):.3f}')

## 5. Final evaluation on unseen people

This is the first use of the held-out test set. Confidence intervals use a cluster bootstrap: whole people are resampled, preserving within-person dependence.

In [ ]:
best_estimator.fit(X_train, y_train)
test_probability = best_estimator.predict_proba(X_test)[:, 1]
test_pred = (test_probability >= decision_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

test_metrics = pd.Series({
    'ROC AUC': roc_auc_score(y_test, test_probability),
    'Average precision': average_precision_score(y_test, test_probability),
    'No-skill AP (test prevalence)': y_test.mean(),
    'Brier loss (lower is better)': brier_score_loss(y_test, test_probability),
    'Balanced accuracy': balanced_accuracy_score(y_test, test_pred),
    'Precision at chosen threshold': precision_score(y_test, test_pred, zero_division=0),
    'Recall at chosen threshold': recall_score(y_test, test_pred),
    'Specificity at chosen threshold': tn / (tn + fp),
    'F1 at chosen threshold': f1_score(y_test, test_pred),
})
display(test_metrics.to_frame(best_name).style.format('{:.3f}'))
display(pd.DataFrame([[tn, fp], [fn, tp]], index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))

# Cluster bootstrap for discrimination metrics.
rng = np.random.default_rng(RANDOM_STATE)
test_frame = pd.DataFrame({'y': y_test.to_numpy(), 'p': test_probability, 'group': g_test.to_numpy()})
group_rows = {key: block.index.to_numpy() for key, block in test_frame.groupby('group')}
unique_groups = np.array(list(group_rows))
boot = []
for _ in range(1_000):
    sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
    sampled_rows = np.concatenate([group_rows[g] for g in sampled_groups])
    y_b = test_frame.loc[sampled_rows, 'y'].to_numpy()
    p_b = test_frame.loc[sampled_rows, 'p'].to_numpy()
    if np.unique(y_b).size == 2:
        boot.append((roc_auc_score(y_b, p_b), average_precision_score(y_b, p_b)))
boot = np.asarray(boot)
ci = pd.DataFrame({
    'estimate': [test_metrics['ROC AUC'], test_metrics['Average precision']],
    '95% CI low': np.quantile(boot, 0.025, axis=0),
    '95% CI high': np.quantile(boot, 0.975, axis=0),
}, index=['ROC AUC', 'Average precision'])
display(ci.style.format('{:.3f}'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[0], name=best_name)
PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[1], name=best_name)
axes[1].axhline(y_test.mean(), color='black', ls='--', lw=1, label='no-skill prevalence')
axes[1].legend()
CalibrationDisplay.from_predictions(y_test, test_probability, n_bins=10, strategy='quantile', ax=axes[2], name=best_name)
plt.tight_layout();

## 6. What drives predictions, and does performance vary by recorded sex?

Permutation importance measures the decrease in test AP after shuffling one original input. It is model-specific and associational, not causal. Subgroup estimates are descriptive and may be noisy; sex is the only supplied demographic category suitable for this small check.

In [ ]:
perm = permutation_importance(
    best_estimator, X_test, y_test, scoring='average_precision',
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=1
)
importance = pd.DataFrame({
    'feature': X_test.columns,
    'AP decrease mean': perm.importances_mean,
    'AP decrease sd': perm.importances_std,
}).sort_values('AP decrease mean', ascending=False)
display(importance)

plot_data = importance.sort_values('AP decrease mean')
plt.figure(figsize=(8, 5))
plt.barh(plot_data.feature, plot_data['AP decrease mean'], xerr=plot_data['AP decrease sd'], color='#4C78A8')
plt.axvline(0, color='black', lw=1)
plt.xlabel('Decrease in average precision after permutation')
plt.title('Permutation importance on held-out people')
plt.tight_layout();

subgroup_rows = []
for value in sorted(X_test['sex'].dropna().unique()):
    mask = X_test['sex'].eq(value).to_numpy()
    y_s, p_s, pred_s = y_test.to_numpy()[mask], test_probability[mask], test_pred[mask]
    subgroup_rows.append({
        'sex': value, 'n': mask.sum(), 'positive_rate': y_s.mean(),
        'roc_auc': roc_auc_score(y_s, p_s) if np.unique(y_s).size == 2 else np.nan,
        'average_precision': average_precision_score(y_s, p_s),
        'precision': precision_score(y_s, pred_s, zero_division=0),
        'recall': recall_score(y_s, pred_s, zero_division=0),
    })
subgroup_results = pd.DataFrame(subgroup_rows).set_index('sex')
display(subgroup_results.style.format({c: '{:.3f}' for c in subgroup_results.columns if c != 'n'}))

## 7. Conclusion generated from the held-out results

In [ ]:
auc = test_metrics['ROC AUC']
ap = test_metrics['Average precision']
base_ap = test_metrics['No-skill AP (test prevalence)']
if auc >= 0.80:
    strength = 'strong'
elif auc >= 0.70:
    strength = 'moderate'
elif auc >= 0.60:
    strength = 'modest'
else:
    strength = 'weak'

print(
    f'Answer: the supplied variables contain {strength} predictive signal for previously unseen people.\n'
    f'The selected {best_name} achieved ROC AUC {auc:.3f} (95% cluster-bootstrap CI '
    f'{ci.loc["ROC AUC", "95% CI low"]:.3f}–{ci.loc["ROC AUC", "95% CI high"]:.3f}) and '
    f'average precision {ap:.3f}, versus a no-skill AP of {base_ap:.3f}.\n'
    f'At the training-selected threshold ({decision_threshold:.3f}), test recall was '
    f'{test_metrics["Recall at chosen threshold"]:.1%} and precision was '
    f'{test_metrics["Precision at chosen threshold"]:.1%}.\n\n'
    'Interpretation: performance above the dummy baseline demonstrates predictability, but the size of the '
    'improvement determines whether it is operationally useful. The present result should be treated as a '
    'risk-ranking prototype, not evidence that the model is ready for clinical use.'
)

### Limitations and next steps

- The data has no admission/discharge date, hospital/site, diagnoses, prior utilization window, comorbidities, or social determinants. Temporal drift and site transportability therefore cannot be tested.
- Grouped validation answers the cold-start question for unseen people. If deployment includes known patients, a separate, carefully time-ordered experiment could add only history available before each discharge (for example, prior admissions/readmissions).
- `birth_year` cannot be converted to age without a calendar year. `postcode_area` may encode socioeconomic or geographic disparities and warrants governance review.
- Threshold selection here illustrates a 70% recall policy. A real threshold needs explicit intervention costs, capacity, and clinical harm analysis.
- Next: obtain dates and richer pre-discharge predictors; perform temporal and external validation; assess calibration and clinically relevant subgroups; compare against an existing clinical score; then prospectively evaluate impact.